# Perplexity, Cross-Entropy, and Compression

A language model's held-out log-loss is usually reported as a perplexity, and perplexity is usually read as a
score. This notebook reads it as a **file size**. An arithmetic coder turns any sequential model into a
compressor whose output is within two bits of $-\log_2 Q(\text{text})$, so cross-entropy, compression, and
perplexity are one quantity in three units. Every number is computed by `perplexity_cross_entropy.py`, which
owns them. Nothing here redefines the math: the notebook imports Topic 3's smoothers, Topic 2's BPE, and the
shared finance corpus.

The arc: cross-entropy and why the log score is the right one (§1); perplexity as a branching factor, made
exact by the AEP (§2); what held-out text estimates, and with what standard error (§3); compression is
prediction — McMillan, the Shannon code, and arithmetic coding on the whole held-out corpus (§4); which unit
makes models comparable across tokenizers, and what `<unk>` hides (§5); and the finance case study on EDGAR
MD&A text (§6).

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "perplexity_cross_entropy.py").exists()
                       else here / "notebooks" / "perplexity-cross-entropy"))
import perplexity_cross_entropy as pc

ng = pc.ng
for t in pc.TESTS:
    s = ng.corpus()[t]
    print(f"{t:5}: {len(s):5d} sentences, {pc.n_word_tokens(s):6d} word tokens (words and </s>), "
          f"{pc.n_chars(s):7d} characters of ' '.join(words) + newline")
print(f"|V| = {len(ng.vocabulary())} (Topic 3's training types, </s>, <unk>); "
      f"the uniform model's log-loss is log|V| = {pc.uniform_log_loss():.4f} nats")

## 1. Cross-entropy and the log score

**Definition 1.** For distributions $p, q$ on a finite set, the **cross-entropy** is
$H(p, q) = -\sum_x p(x)\log q(x)$. A sequential model $Q$ scores a text by its total negative log-likelihood
$\mathrm{NLL} = -\sum_i \log Q(x_i \mid x_{<i})$. Per **unit** (the model's own tokens, characters, or word tokens),
the log-loss is $\mathrm{NLL}/n_{\text{unit}}$ and the **perplexity** is $\mathrm{PP}_{\text{unit}} = \exp(\mathrm{NLL}/n_{\text{unit}})$.
**Bits per character** is $\mathrm{NLL}/(n_{\text{char}} \ln 2)$.

**Proposition 1 (cited: formalML, KL divergence, Prop 1).** $H(p, q) = H(p) + \mathrm{KL}(p \,\|\, q) \ge H(p)$, with
equality iff $q = p$.

**Corollary 1 (the log score is strictly proper).** $\mathbb{E}_p[-\log q(X)]$ is uniquely minimized at $q = p$,
because only the KL term depends on $q$. The linear score $\mathbb{E}_p[q(X)]$ is not proper: the point mass at
the mode of $p$ beats $p$ itself.

**Proposition 2 (what a perplexity is).** (i) $\mathrm{PP} = \bigl(\prod_i 1/q_i\bigr)^{1/N}$, the geometric mean
of the inverse probabilities, which is at most their arithmetic mean. (ii) A model uniform on $k$ symbols has
$\mathrm{PP} = k$ on every text. (iii) $1 \le e^{H(p)} \le |\mathrm{supp}\, p|$. (iv) So $\mathrm{PP} > |V|$ means
the model did worse on this text than the uniform guess over $V$.

In [ ]:
pc.test_gibbs_decomposition()
pc.test_log_score_strictly_proper()
pc.test_perplexity_facts()
pc.test_worse_than_uniform_out_of_domain()
fs = pc.finance_summary()
ll = fs["ikn_held_closed_nats"]
print(f"interpolated KN, held out: {ll:.4f} nats = {ll / pc.LN2:.3f} bits per token, perplexity {pc.perplexity(ll):.2f}")
for name in pc.LAB_SMOOTHERS:
    x = fs[f"{name}_edgar_closed_nats"]
    verdict = "worse" if x > pc.uniform_log_loss() else "better"
    print(f"EDGAR, {name:8}: {x:.4f} nats, perplexity {pc.perplexity(x):7.1f} — {verdict} than uniform "
          f"(perplexity {len(ng.vocabulary())})")

## 2. Perplexity as a branching factor

**Theorem 1 (the AEP).** Let $X_1, \dots, X_n$ be i.i.d. from $p$, with $H = H(p)$ and $\sigma^2 = \mathrm{Var}(-\log p(X))$.
For $\varepsilon > 0$ let $A = \{x^n : |{-\tfrac1n}\log p(x^n) - H| \le \varepsilon\}$. Then

$$
P(A) \ge 1 - \frac{\sigma^2}{n\varepsilon^2}, \qquad |A| \le e^{n(H+\varepsilon)}, \qquad |A| \ge P(A)\, e^{n(H-\varepsilon)},
$$

and for any model $q$, $P\bigl(|{-\tfrac1n}\log q(X^n) - H(p, q)| \le \varepsilon\bigr) \ge 1 - \mathrm{Var}(-\log q(X))/(n\varepsilon^2)$.

*Proof.* $-\log p(X^n)$ is a sum of $n$ i.i.d. terms with mean $H$, so the first bound is Chebyshev. Each typical
sequence has $e^{-n(H+\varepsilon)} \le p(x^n) \le e^{-n(H-\varepsilon)}$, and summing over $A$ gives the other
two. $\blacksquare$

Almost all the probability sits on roughly $e^{nH}$ nearly equiprobable sequences. Text then behaves like a
uniform choice among $e^H$ options per token, and a model behaves like a uniform guess among $e^{H(p,q)}$. Below
the check is exact, by summing over type classes.

In [ ]:
pc.test_typical_set_aep()
ex = pc.typical_example()
p, n, eps = pc.TYPICAL_EXAMPLE
print(f"p = {p}: H = {ex['H']:.4f} nats, e^H = {ex['branching']:.3f} effective choices out of 3")
print(f"n = {n}, eps = {eps}: P(A) = {ex['prob']:.4f}, |A| = e^{ex['log_size']:.1f}, "
      f"a share {ex['share_of_sequences']:.2e} of all 3^{n} sequences")
for n in (40, 120, 300, 1000):
    t = pc.typical_set(p, n, eps)
    print(f"  n = {n:4d}: P(A) = {t['prob']:.4f} (Chebyshev >= {t['chebyshev']:.3f}), (1/n) log|A| = {t['log_size'] / n:.4f}")

## 3. From held-out text to cross-entropy

**Definition 2.** Sentences $S_1, S_2, \dots$ are i.i.d. token sequences ending in `</s>`. Let
$Y_j = -\log Q(S_j)$ and $N_j = |S_j|$. The per-token cross-entropy of $Q$ against the source is
$r = \mathbb{E}[Y]/\mathbb{E}[N]$.

**Theorem 2 (held-out log-loss is a ratio estimator).** If $\mathbb{E}[N], \mathbb{E}[Y] < \infty$, then
$\hat r_m = \sum_j Y_j / \sum_j N_j \to r$ almost surely. With finite second moments,
$\sqrt m(\hat r_m - r) \Rightarrow \mathcal N\bigl(0, \mathrm{Var}(Y - rN)/\mathbb{E}[N]^2\bigr)$, estimated by

$$
\widehat{\mathrm{se}}^2 = \frac{\sum_j (Y_j - \hat r N_j)^2}{m(m-1)\,\bar N^2}.
$$

*Proof.* The strong law applies to $\bar Y$ and $\bar N$ separately, and $\hat r_m = \bar Y / \bar N$. Then
$\hat r_m - r = (\bar Y - r\bar N)/\bar N$, where $\bar Y - r \bar N$ is a mean of i.i.d. zero-mean terms, so the
CLT and Slutsky give the limit law. $\blacksquare$ The unit is the sentence, not the token.

**Shannon–McMillan–Breiman (cited).** For stationary ergodic sources, $-\frac1n \log P(X^n)$ converges almost
surely to the entropy rate. For a finite-order Markov model $Q$, $-\frac1n\log Q(X^n)$ is an ergodic average and
converges to the cross-entropy rate.

**Corollary 2.** $\mathbb{E}[-\log Q(X^n)] = H(X^n) + \mathrm{KL}(P^n \,\|\, Q^n)$, so the cross-entropy rate is
at least the entropy rate: every model estimates an upper bound on the source's entropy.

In [ ]:
pc.test_ratio_estimator_target()
pc.test_ratio_se_coverage()
pc.test_corpus_standard_errors()
a, b = pc.sentence_nll("ikn"), pc.sentence_nll("mkn")
r, se = pc.ratio_estimate(a[:, 0], a[:, 1])
print(f"interpolated KN: {r:.4f} ± {se:.4f} nats/token (sentences as units); "
      f"naive token-level se {pc.token_level_se('ikn'):.4f}")
d, sd = pc.ratio_estimate(b[:, 0] - a[:, 0], a[:, 1])
print(f"modified KN − interpolated KN: {d:.4f} ± {sd:.4f} (paired by sentence) — {abs(d) / sd:.0f} se from 0")
x, sx = pc.ratio_estimate(a[:, 3], a[:, 2])
print(f"open-vocabulary code: {x / pc.LN2:.4f} ± {sx / pc.LN2:.4f} bits per character")

In [ ]:
pc.test_smb_on_a_markov_chain()
pc.test_cross_entropy_rate_bounds_entropy_rate()
rng = np.random.default_rng([pc.SEED, 100])
P, Q = pc.random_chain(rng), pc.random_chain(rng)
x = pc.sample_path(P, 100_000, rng)
print(f"entropy rate {pc.entropy_rate(P):.4f}, cross-entropy rate {pc.cross_entropy_rate(P, Q):.4f} nats/symbol")
for n in (100, 1_000, 10_000, 100_000):
    print(f"  n = {n:6d}: -(1/n) log P(X^n) = {-np.log(P[x[:n - 1], x[1:n]]).mean():.4f}, "
          f"-(1/n) log Q(X^n) = {-np.log(Q[x[:n - 1], x[1:n]]).mean():.4f}")

## 4. Compression is prediction

**Kraft's inequality (cited: formalML, Shannon entropy, Thm 6).** Lengths $\ell_1, \dots, \ell_k$ admit a prefix
code iff $\sum_i 2^{-\ell_i} \le 1$. Codeword $c$ names the dyadic interval $[0.c,\ 0.c + 2^{-|c|})$ of $[0, 1)$. The
prefix property says these intervals are disjoint.

**Theorem 3 (McMillan).** Every uniquely decodable code satisfies $\sum_x 2^{-\ell(x)} \le 1$.

*Proof.* With $K = \sum_x 2^{-\ell(x)}$,

$$
K^m = \sum_{j=1}^{m\ell_{\max}} A_j\, 2^{-j},
$$

where $A_j$ counts the $m$-symbol messages whose concatenated codeword has $j$ bits. Unique decodability makes
those $j$-bit strings distinct, so $A_j \le 2^j$ and $K^m \le m\ell_{\max}$. If $K > 1$ the left side grows
exponentially and the right side linearly. $\blacksquare$ So prefix codes lose nothing. The check below is
exhaustive over small binary codes, with Sardinas–Patterson cross-checked against brute-force parse counting.

In [ ]:
pc.test_kraft_prefix_and_converse()
pc.test_mcmillan_exhaustive()
c = pc.mcmillan_census()
print(f"{c['codes']} binary codes (3 or 4 codewords, length <= 3): {c['ud']} uniquely decodable, "
      f"{c['ud_not_prefix']} of them not prefix-free; {c['kraft_violations']} break Kraft, "
      f"{c['ud_kraft_violations']} of those decodable; brute-force disagreements: {c['brute_force_disagreements']}")
print("{0, 01, 11}:", pc.is_uniquely_decodable(["0", "01", "11"]), "uniquely decodable,",
      pc.is_prefix_free(["0", "01", "11"]), "prefix-free; Kraft sum", pc.kraft_sum([1, 2, 2]))

**Proposition 3 (coding with the wrong model costs KL).** The Shannon code $\ell_q(x) = \lceil -\log_2 q(x)\rceil$
is a prefix code, and

$$
H_2(p) + \mathrm{KL}_2(p\,\|\,q) \;\le\; \mathbb{E}_p[\ell_q] \;<\; H_2(p) + \mathrm{KL}_2(p\,\|\,q) + 1 .
$$

*Proof.* $\sum 2^{-\ell_q} \le \sum q = 1$ (Kraft). Then $-\log_2 q \le \ell_q < -\log_2 q + 1$, and Prop 1 finishes
it. $\blacksquare$

In [ ]:
pc.test_source_coding_with_the_wrong_model()
p_, q_ = np.array([0.5, 0.25, 0.125, 0.125]), np.array([0.25, 0.25, 0.25, 0.25])
print("p =", p_, " Huffman lengths", pc.huffman_lengths(p_), " Shannon lengths for p", pc.shannon_lengths(p_))
print(f"coding p with the uniform q: E[l_q] = {p_ @ np.array(pc.shannon_lengths(q_)):.3f} bits = "
      f"H2(p) {pc.zhv.entropy(p_) / pc.LN2:.3f} + KL2 {pc.kl(p_, q_) / pc.LN2:.3f}")

**Theorem 4 (arithmetic coding).** Code $x = x_1 \dots x_n$ by nested intervals. Start from $[0, 1)$, and at each
step keep the slice of the current interval assigned to $x_i$, of relative width $Q(x_i \mid x_{<i})$. The final
interval $[a, a + Q(x))$ has width $Q(x)$. With $L = \lceil -\log_2 Q(x)\rceil + 1$ and $b = \lceil a 2^L\rceil$:
(i) $[b/2^L, (b+1)/2^L) \subseteq [a, a + Q(x))$; (ii) over any prefix-free set of messages the codewords form a
prefix code; (iii) $-\log_2 Q(x) \le L_{\min} \le \lceil -\log_2 Q(x)\rceil + 1$.

*Proof.* (i) $2^{-L} \le Q(x)/2$ and $b/2^L < a + 2^{-L}$, so $(b+1)/2^L < a + Q(x)$. (ii) Distinct messages have
disjoint intervals, hence disjoint dyadic subintervals. (iii) A dyadic interval inside needs
$2^{-L} \le Q(x)$. $\blacksquare$

The coder uses integer frequencies $f = \lfloor q(2^{40} - |A|)\rfloor + 1$ and exact big-integer rationals:
$A \leftarrow A T + W\,\mathrm{cum}$, $W \leftarrow W f$, $D \leftarrow D T$. Below: one sentence traced symbol by
symbol, then the entire held-out corpus as one codeword.

In [ ]:
pc.test_arithmetic_code_random_instances()
test, words = pc.LAB_SENTENCES[0]
tr = pc.interval_trace(pc.open_model("ikn"), pc.open_model("ikn").symbols([words]))
print(" ".join(words))
for st in tr["steps"]:
    sym, mode, q, lo, bits, settled = st["s"]
    print(f"  {sym:10} Q = {q:.6f}  slice starts at {lo:.6f}  -log2 Q so far {bits:7.3f}  settled bits {settled:3d}  "
          f"alternatives {[t[0] for t in st['top'][:3]]}")
print(f"codeword ({tr['length']} bits, -log2 Q = {tr['model_bits']}, symbol-by-symbol Shannon {tr['shannon_bits']}): "
      f"{tr['codeword']}")

In [ ]:
pc.test_arithmetic_code_corpus()
for t in pc.TESTS:
    code = pc.corpus_code("ikn", t)
    nc = pc.n_chars(ng.corpus()[t])
    print(f"{t:5}: {code.n_symbols} symbols -> ONE codeword of {code.length} bits; -log2 Q = {code.model_bits:.2f}, "
          f"-log2 Q~ = {code.quantized_bits:.2f}; {code.length / nc:.4f} bits per character; "
          f"symbol-by-symbol Shannon codes: {code.shannon_bits} bits")

**Proposition 4 (a compressor is a model).** If $C$ is uniquely decodable, $q_C(x) = 2^{-\ell(x)}$ has
$\sum q_C \le 1$ (Theorem 3), so a file size is the negative log-likelihood of a sub-probability model.
Theorem 4 is the converse. So compressors and language models can be compared in bits per character. For
zlib, "given the training text" is a genuine conditional code (a preset dictionary). For bz2 and lzma it is the
difference of file sizes $|C(\text{train} + \text{test})| - |C(\text{train})|$.

In [ ]:
pc.test_compressors_are_models()
comp = pc.compressor_table()
print(f"{'bits per character':20} {'held, alone':>12} {'held, given':>12} {'EDGAR, alone':>13} {'EDGAR, given':>13}")
for name, row in comp.items():
    print(f"{name:20} {row['held']['alone']:12.4f} {row['held']['given_train']:12.4f} "
          f"{row['edgar']['alone']:13.4f} {row['edgar']['given_train']:13.4f}")
for t in pc.TESTS:
    print(f"interpolated KN code, {t}: {pc.corpus_code('ikn', t).length / pc.n_chars(ng.corpus()[t]):.4f} bits per character")

## 5. Choosing the unit

**Theorem 5 (tokenization invariance).** Let $P$ be a distribution on strings, and $\tau$ an injective tokenizer.
The token model $Q_\tau(t) = P(\tau^{-1}(t))$ has $\log Q_\tau(\tau(s)) = \log P(s)$. So for a text $T$ and any two
units, $n_A \log \mathrm{PP}_A = n_B \log \mathrm{PP}_B = -\log P(T)$, and every tokenizer-free unit ranks models
identically.

*A word model is a character model.* Let $Q'$ be interpolated KN without `<unk>`, renormalized. After context
$u$, the prefix mass $M(\pi) = \sum_{w \sqsupseteq \pi} Q'(w \mid u)$ is a contiguous slice of the sorted
vocabulary. Letter $c$ has probability $M(\pi c)/M(\pi)$. The separator after a complete word $w$ has probability
$\frac{Q'(w \mid u)}{M(w)}$ times $Q'(\texttt{</s>}\text{ next})$ for the sentence end, or $1 - Q'(\texttt{</s>}\text{ next})$
for a space. Each word's factors telescope to $Q'(w \mid u)$.

In [ ]:
pc.test_vector_models_match_scalar()
pc.test_tokenization_invariance()
d = pc.invariance_demo()
print(f"Q' on {d['sentences']} in-vocabulary held-out sentences: {d['nats']:.2f} nats in all = "
      f"perplexity {d['pp_word']:.2f} per word token ({d['word_tokens']}) = {d['pp_char']:.4f} per character ({d['chars']})")
vec = pc.word_vector("ikn")
s = pc.LAB_SENTENCES[0][1]
lc = pc.induced_char_logprobs(s, vec)
text = " ".join(s) + "⏎"
print("per-character -log2 Q':", " ".join(f"{ch}:{0.0 - x / pc.LN2:.2f}" for ch, x in zip(text[:24], lc[:24])), "…")
print(f"character total {-sum(lc) / pc.LN2:.6f} bits = word total {-pc.word_logprob(s, vec) / pc.LN2:.6f} bits")

In [ ]:
pc.test_per_token_perplexity_misranks()
print(f"{'model (held out)':14} {'tok/char':>9} {'nats/tok':>9} {'PP/token':>9} {'bpc':>7} {'PP/word':>10}")
for name in pc.unit_models():
    r = pc.unit_row(name, "held")
    flag = "" if r["is_code"] else "   <- not a code"
    print(f"{name:14} {r['tokens_per_char']:9.4f} {r['nats_per_token']:9.4f} {r['pp_token']:9.3f} {r['bpc']:7.4f} "
          f"{r['pp_word']:10.2f}{flag}")
print("BPE vocabulary at Topic 2's MDL optimum:", pc.bpe_mdl_size())

**Proposition 5 (`<unk>` can only flatter).** If an open completion spells each OOV word with
$S(w \mid \cdot)$, where $\sum_w S \le 1$, then

$$
-\log Q_{\text{open}}(\text{text}) = -\log Q_{\text{closed}}(\text{text}) - \sum_{\text{OOV}} \log S \;\ge\; -\log Q_{\text{closed}}(\text{text}) .
$$

The closed score is not a code length, because it gives one probability to infinitely many strings. Below,
every word with training count $\le k$ becomes `<unk>`.

In [ ]:
pc.test_unk_flatters()
for t in pc.TESTS:
    print(f"{t}: {'k':>4} {'|V_k|':>6} {'OOV':>7} {'closed PP':>10} {'open bpc':>9}")
    for k in pc.UNK_CUTOFFS:
        r = pc.cutoff_row(k, t)
        print(f"{'':6}{'all' if r['k'] is None else r['k']:>4} {r['V']:6d} {r['oov_share']:7.2%} "
              f"{r['closed_pp']:10.2f} {r['open_bpc']:9.4f}")

## 6. Finance case study: EDGAR MD&A

Topic 3 found that out of domain (EDGAR excerpts, 57.59% of tokens OOV) Laplace beats Kneser–Ney. Here the same
reversal is read three ways. As a branching factor, interpolated KN does worse than uniform. In honest units,
most of the code is spelling, which the `<unk>` convention hides. Against a general-purpose compressor, lzma,
adapting to the text it codes, beats every static model. Even the speller, fit to the corpus's planted
pseudo-words, loses to plain letter frequencies on real English.

In [ ]:
pc.test_finance_findings()
fs = pc.finance_summary()
print(f"OOV share: held out {fs['oov_share_held']:.2%}, EDGAR {fs['oov_share_edgar']:.2%}")
for name in pc.LAB_SMOOTHERS:
    print(f"{name:8} EDGAR: closed {fs[f'{name}_edgar_closed_nats']:.4f} nats/token ({fs[f'{name}_edgar_closed_bpc']:.4f} 'bpc'), "
          f"open {fs[f'{name}_edgar_open_bpc']:.4f} bpc")
print(f"share of the honest code that is spelling: EDGAR {fs['spelled_share_edgar']:.1%}, held out {fs['spelled_share_held']:.1%}")
for t, row in pc.speller_comparison().items():
    print(f"{t:5}: open bpc with a letter-unigram speller {row[1]:.4f}, with the trigram speller {row[pc.SPELL_ORDER]:.4f}")
print(f"lzma given the training text, EDGAR: {pc.compressor_table()['lzma']['edgar']['given_train']:.4f} bpc")

## 7. The lab's numbers, and the full harness

`viz_payload()` holds every number the interactive lab shows. Panel A has three sentences, each traced
symbol by symbol under Laplace, interpolated KN, and modified KN: slices, alternatives, bits, and settled bits.
Panel B has the unit table and the `<unk>` cutoff curves for both test sets, plus the corpus codes and the
compressor table.

In [ ]:
pc.test_viz_payload()
P = pc.viz_payload()
for sent in P["sentences"]:
    print(" ".join(sent["words"]), f"({sent['test']}, {sent['chars']} chars):",
          {k: v["length"] for k, v in sent["by_smoother"].items()}, "bits")
print("corpus codes:", P["codes"])

In [ ]:
assert pc._run_harness() == 0